In [ ]:
import pandas as pd
import re
# pd.set_option('display.width', 1000)
df = pd.read_csv(r"securitytestlogs.csv", engine='python', sep=',', on_bad_lines='skip')
df.reset_index(drop=True, inplace=True)

# print(df.head().to_string())

In [362]:
print(df.columns.tolist())


['Level', 'Date and Time', 'Source', 'Event ID', 'Task Category', 'Message']


In [363]:
#Creates new column account name and extracts from task category column
def extractName(text):
    match = re.findall(r"Account Name:\s*(\w+)",text)

    return match[-1] if match else None #Last match is the relevant one

df['accountName'] = df['Message'].apply(extractName).tolist()


In [364]:
success=(df[df['Event ID']==4624]) #success logins
failed=(df[df['Event ID']==4625]) #failed logins

print(failed['accountName'].value_counts())

fakeuser    13
Maryam       2
Name: accountName, dtype: int64


In [365]:
#Prints unique usernames that have failed more than 3 times
for username in failed['accountName'].unique():
    if (failed['accountName']==username).sum()>=3:
        print("Repeated failed login attempts, username is: ",username)


Repeated failed login attempts, username is:  fakeuser


In [366]:
#Prints unique usernames that have failed more than 3 times within 5 minutes 

failed['Date and Time'] = pd.to_datetime(failed['Date and Time']) #Date and time is an object type, needs casting 

failed=failed.sort_values('Date and Time') #Orders chronologically 

#Rolling window of 5 minutes - needs time as the index
failed=failed.set_index('Date and Time')

failed['failCountIn5Mins'] = (
    failed
    .groupby('accountName')['Event ID']
    .rolling('5min')
    .count()
    .reset_index(level=0, drop=True)
    .to_numpy()
)

flagged = failed[failed['failCountIn5Mins'] >= 3] #Gathers only failed login attempts of concern

alertsLog=[]
for username,count in (flagged['accountName'].value_counts()).items():
    if count>=3 and count<=5:
        alertsLog.append({
            "username":username,
            "count":count,
            "severity":"Low"
        })
    elif count>5 and count<=7:
        alertsLog.append({
            "username":username,
            "count":count,
            "severity":"Medium"
        })
    elif count>7:
        alertsLog.append({
            "username":username,
            "count":count,
            "severity":"High"
        })

print(alertsLog)

[{'username': 'fakeuser', 'count': 4, 'severity': 'Low'}]


C:\Users\mgadi\AppData\Local\Temp\ipykernel_8336\2944870242.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  failed['Date and Time'] = pd.to_datetime(failed['Date and Time']) #Date and time is an object type, needs casting


In [ ]:
#Writes the alerts log to file, only includes columns: Date and Time,Level,Source,Event ID,Task Category,Message,accountName

failed.iloc[:,:6].to_csv(r'alert.csv',mode='a')